# Watermark Comparison Demo

Compare original video with VideoSeal/Adaptive VideoSeal results frame by frame to visualize watermark embedding effects.


In [ ]:
from pathlib import Path
from typing import Tuple, List
import subprocess
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import platform
from matplotlib import font_manager


def _configure_korean_font() -> None:
    """Set a Korean-friendly font based on the current OS."""
    font_candidates = {
        "Darwin": ["AppleGothic", "NanumGothic"],
        "Windows": ["Malgun Gothic", "Nanum Gothic"],
        "Linux": ["NanumGothic", "Noto Sans CJK KR", "UnDotum"],
    }
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in font_candidates.get(platform.system(), []):
        if name in installed:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False


_configure_korean_font()



In [ ]:
def get_video_frame_count(video_path: Path) -> int:
    """Get actual frame count using ffprobe -show_frames"""
    try:
        cmd = [
            'ffprobe', '-v', 'error',
            '-show_frames', '-select_streams', 'v:0',
            '-of', 'json', str(video_path)
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        data = json.loads(result.stdout)
        if 'frames' in data:
            return len(data['frames'])
        return 0
    except (subprocess.CalledProcessError, json.JSONDecodeError, KeyError) as e:
        # fallback: use cv2
        cap = cv2.VideoCapture(str(video_path))
        if cap.isOpened():
            count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            return count
        raise FileNotFoundError(f"Unable to open {video_path}: {e}")


def load_frame(video_path: Path, frame_index: int) -> Tuple[np.ndarray, int]:
    # Get actual frame count using ffprobe
    total_frames = get_video_frame_count(video_path)
    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Unable to open {video_path}")

    safe_index = max(0, min(frame_index, total_frames - 1))
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, safe_index)
    ok, frame = cap.read()
    cap.release()

    if not ok:
        raise ValueError(f"Failed to read frame {safe_index} from {video_path} (total: {total_frames})")

    return frame, total_frames


def to_rgb(frame: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _diff_to_gray(diff: np.ndarray, scale: float = 6.0) -> np.ndarray:
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY).astype(np.float32) * scale
    return np.clip(gray, 0, 255).astype(np.uint8)


def diff_colormap(diff: np.ndarray, cmap: str = "inferno", scale: float = 6.0) -> np.ndarray:
    gray = _diff_to_gray(diff, scale)
    normalized = gray.astype(np.float32) / 255.0
    colormap = plt.get_cmap(cmap)
    return (colormap(normalized)[..., :3] * 255).astype(np.uint8)


def diff_grayscale(diff: np.ndarray, scale: float = 6.0) -> np.ndarray:
    return _diff_to_gray(diff, scale)


def load_frames_range(video_path: Path, center_index: int, num_frames: int = 5) -> Tuple[List[np.ndarray], int]:
    """Load frames around center frame (before and after)"""
    total_frames = get_video_frame_count(video_path)
    
    # Calculate frame range (2 frames before and after center)
    half_range = num_frames // 2
    start_index = max(0, center_index - half_range)
    end_index = min(total_frames - 1, center_index + half_range)
    
    # Actual frame indices to load
    frame_indices = list(range(start_index, end_index + 1))
    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Unable to open {video_path}")
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            raise ValueError(f"Failed to read frame {idx} from {video_path}")
        frames.append(frame)
    
    cap.release()
    return frames, total_frames


def show_frames_range(
    original_frames: List[np.ndarray],
    videoseal_frames: List[np.ndarray],
    adaptive_frames: List[np.ndarray],
    frame_indices: List[int],
    cmap: str = "inferno",
    scale: float = 6.0,
    show_watermark: bool = True,
) -> None:
    """Display multiple frames in 5xN grid"""
    num_frames = len(original_frames)
    if num_frames != len(videoseal_frames) or num_frames != len(adaptive_frames):
        raise ValueError("All frame lists must have the same length")
    
    # Always 5 rows (watermark shown as absolute value only)
    num_rows = 5
    
    # Row titles
    row_titles = ["Original", "VideoSeal", "Adaptive", "VS Watermark", "Ad Watermark"]
    
    # Use GridSpec to add label column
    from matplotlib.gridspec import GridSpec
    fig = plt.figure(figsize=(2.2 * (num_frames + 0.3), 2.0 * num_rows))
    gs = GridSpec(num_rows, num_frames + 1, figure=fig, 
                  width_ratios=[0.15] + [1] * num_frames,
                  wspace=0.02, hspace=0.08, left=0.05, right=0.98, top=0.95, bottom=0.05)
    
    # Create label column
    label_axes = []
    for row_idx in range(num_rows):
        ax = fig.add_subplot(gs[row_idx, 0])
        ax.text(1.0, 0.5, row_titles[row_idx], 
                transform=ax.transAxes,
                fontsize=11, va='center', ha='right', weight='bold')
        ax.axis('off')
        label_axes.append(ax)
    
    # Create image columns
    axes = []
    for row_idx in range(num_rows):
        row_axes = []
        for col_idx in range(num_frames):
            ax = fig.add_subplot(gs[row_idx, col_idx + 1])
            row_axes.append(ax)
        axes.append(row_axes)
    
    for col_idx in range(num_frames):
        orig = original_frames[col_idx]
        vs = videoseal_frames[col_idx]
        ad = adaptive_frames[col_idx]
        
        # Calculate differences
        diff_vs = cv2.absdiff(vs, orig)
        diff_ad = cv2.absdiff(ad, orig)
        
        # Display each row
        # Row 0: Original
        axes[0][col_idx].imshow(to_rgb(orig))
        axes[0][col_idx].set_title(f"F{frame_indices[col_idx]}", fontsize=8, pad=1)
        axes[0][col_idx].axis("off")
        
        # Row 1: VideoSeal
        axes[1][col_idx].imshow(to_rgb(vs))
        axes[1][col_idx].axis("off")
        
        # Row 2: Adaptive
        axes[2][col_idx].imshow(to_rgb(ad))
        axes[2][col_idx].axis("off")
        
        # Row 3: VS Watermark (color heatmap)
        axes[3][col_idx].imshow(diff_colormap(diff_vs, cmap=cmap, scale=scale))
        axes[3][col_idx].axis("off")
        
        # Row 4: Ad Watermark (color heatmap)
        axes[4][col_idx].imshow(diff_colormap(diff_ad, cmap=cmap, scale=scale))
        axes[4][col_idx].axis("off")
    
    plt.show()


def show_watermark_comparison(
    frame_index: int,
    original_video: Path,
    videoseal_video: Path,
    adaptive_video: Path,
    num_frames: int = 5,
    cmap: str = "inferno",
    scale: float = 6.0,
) -> None:
    """Integrated function to display watermark comparison results for given frame index"""
    # Get frame count for each video
    orig_total = get_video_frame_count(original_video)
    vs_total = get_video_frame_count(videoseal_video)
    ad_total = get_video_frame_count(adaptive_video)
    
    # Adjust frame index (considering max frame count of each video)
    max_frames = min(orig_total, vs_total, ad_total)
    adjusted_index = min(frame_index, max_frames - 1)
    adjusted_index = max(0, adjusted_index)
    
    # Check and print if adjusted
    if frame_index != adjusted_index:
        print(f"⚠️ Frame index adjusted: {frame_index} → {adjusted_index} (max frames: {max_frames})")
    
    # Calculate frame range (same range for all videos)
    half_range = num_frames // 2
    start_idx = max(0, adjusted_index - half_range)
    end_idx = min(max_frames - 1, adjusted_index + half_range)
    frame_indices = list(range(start_idx, end_idx + 1))
    
    # Load frames from same index range for each video
    def load_frames_by_indices(video_path: Path, indices: List[int]) -> List[np.ndarray]:
        """Load frames for given frame index list"""
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise FileNotFoundError(f"Unable to open {video_path}")
        
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            if not ok:
                raise ValueError(f"Failed to read frame {idx} from {video_path}")
            frames.append(frame)
        
        cap.release()
        return frames
    
    # Load frames from same index range for all videos
    orig_frames = load_frames_by_indices(original_video, frame_indices)
    vs_frames = load_frames_by_indices(videoseal_video, frame_indices)
    ad_frames = load_frames_by_indices(adaptive_video, frame_indices)
    
    # Print information
    print(f"Frame index: {adjusted_index} (center)")
    print(f"Display range: {frame_indices[0]} ~ {frame_indices[-1]} ({len(frame_indices)} frames)")
    print(f"Original frames: {orig_total}")
    print(f"VideoSeal frames: {vs_total}")
    print(f"Adaptive frames: {ad_total}")
    
    # Display multiple frames
    show_frames_range(orig_frames, vs_frames, ad_frames, frame_indices, cmap=cmap, scale=scale)



In [ ]:
# Path configuration
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

ORIGINAL_VIDEO = DATA_DIR / "sample_movie.mp4"
VIDEOSEAL_VIDEO = OUTPUT_DIR / "trial1_sample_movie.mp4"
ADAPTIVE_VIDEO = OUTPUT_DIR / "adaptive_trial1_sample_movie.mp4"

DEFAULT_FRAME_INDEX = 100  # Default frame index to check

# Check max frame count of VideoSeal video and auto-adjust (using ffprobe)
original_index = DEFAULT_FRAME_INDEX
if VIDEOSEAL_VIDEO.exists():
    vs_max_frames = get_video_frame_count(VIDEOSEAL_VIDEO)
    if vs_max_frames > 0 and DEFAULT_FRAME_INDEX >= vs_max_frames:
        DEFAULT_FRAME_INDEX = vs_max_frames - 1
        if original_index != DEFAULT_FRAME_INDEX:
            print(f"⚠️ Frame index adjusted: {original_index} → {DEFAULT_FRAME_INDEX} (max frames: {vs_max_frames})")



In [ ]:
# Display watermark comparison results
show_watermark_comparison(
    frame_index=DEFAULT_FRAME_INDEX,
    original_video=ORIGINAL_VIDEO,
    videoseal_video=VIDEOSEAL_VIDEO,
    adaptive_video=ADAPTIVE_VIDEO,
    num_frames=5,
    cmap="inferno",
    scale=6.0
)
